In [1]:
# --! include root folder into PYTHONPATH --!

import os
import sys

thisdir = os.getcwd()
rootdir = os.path.abspath(os.path.join(thisdir, '..', '..'))
sys.path.append(rootdir)

# --! import Python libraries --!

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

import torch
import numpy as np
import matplotlib.pyplot as plt

import util_data

In [2]:
# --! sb3 expects a vectorized environment
env = make_vec_env("Hopper-v5", n_envs=4)

AttributeError: module 'mujoco' has no attribute '__version__'

In [ ]:
load_pretrained = True

if not load_pretrained:
    # --! initialize PPO agent
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-4, n_steps=2048, batch_size=64, n_epochs=10)

    # --! train agent
    model.learn(total_timesteps=1_000_000)

    # --! save trained model
    model.save("../../models/mujoco/ppo_hopper_v5")

In [ ]:
# --! load and evaluate

dataloaded = True
data_nsample = 1_000
skip_nsample = 0

if not dataloaded:
    states = []
    actions = []

    model = PPO.load("../../models/mujoco/ppo_hopper_v5")
    obs = env.reset()
    for _ in range(data_nsample):
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, info = env.step(action)
        env.render("human")

        states.append(obs[:1])
        actions.append(action[:1])

    states = np.concatenate(states)
    actions = np.concatenate(actions)

    saved_states = np.expand_dims(states, 0)
    saved_actions = np.expand_dims(actions, 0)
    saved_data = np.concatenate([saved_states, saved_actions], axis=-1)
    saved_data = saved_data[:, skip_nsample:]
    print(f'saved data shape is {saved_data.shape}')

    util_data.write_datafile('../../data/mujoco/hopper_all_0', saved_data)

data = util_data.read_datafile('../../data/mujoco/hopper_all_0', data_nsample - skip_nsample)
print(f'read data shape: {data.shape}')

state_ndim = 11
action_ndim = 3
states, actions = torch.split(data, [state_ndim, action_ndim], dim=-1)
print(f'read states shape: {states.shape}')
print(f'read actions shape: {actions.shape}')

In [ ]:
print(f'states shape: {states.shape}')
print(f'actions shape: {actions.shape}')

disp_end = 1_000

plt.figure(figsize=(6,33))

plt.subplot(11,1,1)
plt.title('states')
plt.plot(states[0, :disp_end, 0], label='z')
plt.legend()

plt.subplot(11,1,2)
plt.plot(states[0, :disp_end, 1])

plt.subplot(11,1,3)
plt.plot(states[0, :disp_end, 2])

plt.subplot(11,1,4)
plt.plot(states[0, :disp_end, 3])

plt.subplot(11,1,5)
plt.plot(states[0, :disp_end, 4])

plt.subplot(11,1,6)
plt.plot(states[0, :disp_end, 5], label='x velocity')
plt.legend()

plt.subplot(11,1,7)
plt.plot(states[0, :disp_end, 6], label='z velocity')
plt.legend()

plt.subplot(11,1,8)
plt.plot(states[0, :disp_end, 7])

plt.subplot(11,1,9)
plt.plot(states[0, :disp_end, 8])

plt.subplot(11,1,10)
plt.plot(states[0, :disp_end, 9])

plt.subplot(11,1,11)
plt.plot(states[0, :disp_end, 10])

plt.show()

plt.figure(figsize=(6,9))

plt.subplot(3,1,1)
plt.title('actions')
plt.plot(actions[0, :disp_end, 0])

plt.subplot(3,1,2)
plt.plot(actions[0, :disp_end, 1])

plt.subplot(3,1,3)
plt.plot(actions[0, :disp_end, 2])

plt.show()

In [ ]:
disp_end = 350

window_fore = 16
window_back = 2 * window_fore
print(f'fore samples number: {window_fore}')
print(f'back samples number: {window_back}')
window_nsample = window_back + window_fore
print(f'window samples number: {window_nsample}')

start_exc = 187 # start=187
end_exc = 260 # end=260
print(f'excursion window length: {end_exc - start_exc}')

window_start = start_exc#end_exc - window_nsample
window_mid = window_start + window_back
window_end = window_start + window_nsample

with torch.no_grad():
    plt.figure(figsize=(6,9))

    plt.subplot(3,1,1)
    plt.plot(states[0, :disp_end, 0], label='z pos')
    plt.plot([window_start,window_start],[1.1,1.6], linestyle='dashed', color='gray')
    plt.plot([window_end,window_end],[1.1,1.6], linestyle='dashed', color='gray')
    plt.plot([window_mid,window_mid],[1.1,1.6], linestyle='dashed', color='gray')
    plt.legend()

    plt.subplot(3,1,2)
    plt.plot(states[0, :disp_end, 6], label='z vel')
    plt.plot([window_start,window_start],[-3,3], linestyle='dashed', color='gray')
    plt.plot([window_end,window_end],[-3,3], linestyle='dashed', color='gray')
    plt.plot([window_mid,window_mid],[-3,3], linestyle='dashed', color='gray')

    plt.subplot(3,1,3)
    plt.plot(states[0, :disp_end, 1])
    plt.plot([window_start,window_start],[-0.1,0.1], linestyle='dashed', color='gray')
    plt.plot([window_end,window_end],[-0.1,0.1], linestyle='dashed', color='gray')
    plt.plot([window_mid,window_mid],[-0.1,0.1], linestyle='dashed', color='gray')

    plt.show()

In [ ]:
def nominal_criterion(sample):
    return sample > 1.3

def excursion_criterion(sample):
    return sample < 1.3

def extract_windows(timeseries, criterion, back_nsample, fore_nsample):

    windows = []

    timeseries_nsample = timeseries.shape[1]
    for i in range(timeseries_nsample):
        if criterion(timeseries[0, i, 0]):
            window_start = i - back_nsample
            window_end = i + fore_nsample
            if window_start >= 0 and window_end <= timeseries_nsample:
                window = timeseries[:, window_start:window_end, :]
                windows.append(window)

    return torch.cat(windows, dim=0)

data_nom = extract_windows(data, nominal_criterion, window_back, window_fore)
data_exc = extract_windows(data, excursion_criterion, window_back, window_fore)
print(data_nom.shape)
print(data_exc.shape)

In [ ]:
datasaved = False

if datasaved:
    util_data.write_datafile('../../data/mujoco/hopper_nom_0', data_nom)
    util_data.write_datafile('../../data/mujoco/hopper_exc_0', data_exc)